In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from openai import OpenAI
from pydantic import BaseModel
from PIL import Image
import numpy as np
import base64
import io

import sys
import os
os.chdir('/home/jhpark/image-artifacts/src')
from pipeline.prompts import MoneyManager

client = OpenAI()
money_manager = MoneyManager(model='gpt-4o')

ARTIFACT_TYPE = 'fusion'

In [4]:
class ArtifactSuccessResponse(BaseModel):
    success: bool
    reasoning: str

In [5]:
def encode_image_to_base64(image):
    """Convert PIL Image or numpy array to base64 string"""
    if isinstance(image, np.ndarray):
        # Convert numpy array to PIL Image
        pil_image = Image.fromarray(image)
    else:
        pil_image = image
    
    # Convert to RGB if necessary
    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')
    
    # Save to bytes buffer
    buffer = io.BytesIO()
    pil_image.save(buffer, format='JPEG')
    buffer.seek(0)
    
    # Encode to base64
    return base64.b64encode(buffer.getvalue()).decode('utf-8')
        

In [6]:
object_name = "an ear of an elephant"

instruction  = {
    "addition": (
        "You are an expert at detecting addition-type artifacts in AI-generated images.\n\n"
        "Addition artifacts occur when a part of an object is duplicated and placed adjacent to the original, "
        "creating anatomically or structurally implausible duplications (e.g., extra fingers, duplicate ears, duplicate wheels). "
        "Your role is to determine if the artifact was successfully injected into the original image on the target region.\n\n"
        "You will be shown:\n"
        "\t1. An original image without the target region\n"
        "\t2. An original image with only the target region\n"
        "\t3. An artifact image with only the target region\n"
        "\t4. The object name that will be added\n"
        f'Your task is to determine if there is the object present in the 3rd image, in the target region.\n\n'
        "Look carefully at the target region and determine:\n"
        f'\t• Is there a clear presence of the object within the target area?\n'
        f'\t• Does it appear to be a plausible duplication/addition of the object?\n\n'
        "Consider that addition artifacts should:\n"
        f'\t• Show the specified object in the masked region\n'
        f'\t• Appear anatomically/structurally similar to other instances of the same object\n'
        f'\t• Be positioned adjacent to where such objects would naturally occur'
    ),
    "removal": (
        "You are an expert at detecting removal-type artifacts in AI-generated images.\n\n"
        "Removal artifacts occur when a part of an object is deleted and the area is inpainted with background, "
        "resulting in missing features or gaps where something should be present (e.g., missing fingers, absent ears). "
        "Your role is to determine if the artifact was successfully injected into the original image on the target region.\n\n"
        "You will be shown:\n"
        "\t1. An original image without the target region\n"
        "\t2. An original image with only the target region\n"
        "\t3. An artifact image with only the target region\n"
        "\t4. The object name that will be removed\n"
        f'Your task is to determine if the object is absent in the 3rd image, in the target region.\n\n'
        "Look carefully at the target region and determine:\n"
        f"\t• Is the expected object clearly missing from the masked area?\n"
        "\t• Do you observe background fill or inpainting traces where the part should be?\n\n"
        "Consider that removal artifacts should:\n"
        f"\t• Show the object missing in the masked region\n"
        "\t• Replace the expected structure with background textures or inpainting\n"
        "\t• Keep surrounding anatomy/structure present but incomplete"
    ),
    "fusion": (
        "You are an expert at detecting fusion-type artifacts in AI-generated images.\n\n"
        "Fusion artifacts occur when a part or two distinct entities are unnaturally merged together, "
        "creating blurred boundaries, overlapped textures, or structural entanglement that makes the separation implausible "
        "(e.g., two animals merged into one, a limb merged into the torso, overlapping facial features). "
        "Your role is to determine if the artifact was successfully injected into the original image on the target region.\n\n"
        
        "You will be shown:\n"
        "\t1. An original image without the target region\n"
        "\t2. An original image with only the target region\n"
        "\t3. An artifact image with only the target region\n"
        "\t4. The two object names that will be fused\n"
        f'Your task is to determine if there is an unnatural fusion involving the two objects present in the 3rd image, in the target region.\n\n'
        "Look carefully at the target region and determine:\n"
        f"\t• Are there clear signs that the two objects are merged with adjacent parts or entities?\n"
        "\t• Do you observe blurred boundaries, overlapped textures, or interpenetration where separation should exist?\n\n"
        "Consider that fusion artifacts should:\n"
        f"\t• Involve the two objects showing unnatural merging with nearby structures in the masked region\n"
        "\t• Display indistinct boundaries or texture blending between elements that should remain separate\n"
        "\t• Make it difficult to tell where one part ends and another begins"
    )
}


prompt = f"""
    {instruction[ARTIFACT_TYPE]}
    Return your analysis in the following JSON format:
    {{
        "reasoning": "Brief explanation of what you observe in the masked region from the third image",
        "success": true/false
    }}
    """

In [7]:
positive_original_masked = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/positive/original_masked.png'))
positive_original_target = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/positive/original_target.png'))
positive_artifact_target = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/positive/artifact_target.png'))
negative_original_masked = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/negative/original_masked.png'))
negative_original_target = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/negative/original_target.png'))
negative_artifact_target = encode_image_to_base64(Image.open(f'pipeline/in_context_exps/{ARTIFACT_TYPE}/negative/artifact_target.png'))
test_original_masked = encode_image_to_base64(Image.open('notebooks/examples/original_masked_1.png'))
test_original_target = encode_image_to_base64(Image.open('notebooks/examples/original_target_1.png'))
test_artifact_target = encode_image_to_base64(Image.open('notebooks/examples/artifact_target_1.png'))
image1 = encode_image_to_base64(Image.open('pipeline/in_context_exps/order_test/1.png'))
image2 = encode_image_to_base64(Image.open('pipeline/in_context_exps/order_test/2.png'))
image3 = encode_image_to_base64(Image.open('pipeline/in_context_exps/order_test/3.png'))
image4 = encode_image_to_base64(Image.open('pipeline/in_context_exps/order_test/4.png'))


In [8]:
reasoning = {
    "addition": {
        "positive": {
            "reasoning": "The third image in the target region shows a clear presence of an elephant's ear. The pattern and texture are consistent with an elephant's ear and are positioned in a manner that is anatomically plausible for an elephant",
            "object_name": "an ear of an elephant"
        },
        "negative": {
            "reasoning": "The third image in the target region does not show a clear presence of a zebra's tail. Although the target region is altered, the pattern and texture are not consistent with a zebra's tail and are not positioned in a manner that is anatomically plausible for a zebra",
            "object_name": "a tail of a zebra"
        }
    },
    "removal": {
        "positive": {
            "reasoning": "The third image shows the region where the zebra's leg is expected. In this image, the leg is absent, and the area is filled with background textures that blend with the ground and surroundings.",
            "object_name": "a leg of a zebra"
        },
        "negative": {
            "reasoning": "The third image in the target region shows where an ear of a cat originally was. In this image, although the ear looks shrinked, compared to the second image, which is the target region of the original image, the object was not removed properly.",
            "object_name": "an ear of a cat"
        }
    },
    "fusion": {
        "positive": {
            "reasoning": "The second image shows two heads of sheeps. In the third image, the heads are fused together, and the two sheeps are merged into one.",
            "object_name": "a sheep and another sheep"
        },
        "negative": {
            "reasoning": "The second image shows two elephants overlapping in the image. In the third image, although the target region looks blurred, the two elephants have a clear boundary.",
            "object_name": "an elephant and a baby elephant"
        }
    }
}

In [16]:
import time

num_runs = 10
total_time = 0.0
responses = []

for _ in range(num_runs):
    start_time = time.time()
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image1}"}},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image2}"}},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image3}"}},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image4}"}},
                    {"type": "text", "text": "describe only the fourth image provided"}
                ]
            },
        ],
    )
    elapsed = time.time() - start_time
    total_time += elapsed
    responses.append(response)

average_time = total_time / num_runs
print(f"Average run time over {num_runs} runs: {average_time:.4f} seconds")

Average run time over 10 runs: 5.0797 seconds


In [17]:
import time

num_runs = 10
total_time = 0.0
responses = []

for _ in range(num_runs):
    start_time = time.time()
    response = client.responses.parse(
        model='gpt-4o',
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image1}"},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image2}"},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image3}"},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image4}"},
                    {"type": "input_text", "text": "describe only the fourth image provided"}
                ]
            }
        ]
    )
    elapsed = time.time() - start_time
    total_time += elapsed
    responses.append(response)

average_time = total_time / num_runs
print(f"Average run time over {num_runs} runs: {average_time:.4f} seconds")

Average run time over 10 runs: 5.8969 seconds


# In context learning template

In [ ]:
response = client.responses.parse(
    model='gpt-4o',
    input=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": [
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{positive_original_masked}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{positive_original_target}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{positive_artifact_target}"},
                        {"type": "input_text", "text": f"{reasoning[ARTIFACT_TYPE]['positive']['object_name']}"}
                    ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "output_text", "text": f'{{"reasoning": "{reasoning[ARTIFACT_TYPE]["positive"]["reasoning"]}", "success": true}}'}
            ]
        },
        {
            "role": "user",
            "content": [
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{negative_original_masked}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{negative_original_target}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{negative_artifact_target}"},
                        {"type": "input_text", "text": f"{reasoning[ARTIFACT_TYPE]['negative']['object_name']}"}
                    ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "output_text", "text": f'{{"reasoning": "{reasoning[ARTIFACT_TYPE]["negative"]["reasoning"]}", "success": false}}'}
            ]
        },
        {
            "role": "user",
            "content": [
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{test_original_masked}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{test_original_target}"},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{test_artifact_target}"},
                        {"type": "input_text", "text": "a sheep and a sheep"}
                    ]
        }
    ],
    text_format=ArtifactSuccessResponse,
)
money_manager.refresh()
money_manager(response)
print(money_manager.total_cost)

BadRequestError: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

In [ ]:
import json
result = json.loads(response.choices[0].message.content)
print(result["reasoning"])
print(result["success"])

In the third image, the head of one sheep appears unnaturally merged with the neck of another, showing blurred textures and indistinct boundaries.
True


In [ ]:
success_count = 0
total_runs = 10
for _ in range(total_runs):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{positive_original_masked}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{positive_original_target}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{positive_artifact_target}"}},
                            {"type": "text", "text": f"{reasoning[ARTIFACT_TYPE]['positive']['object_name']}"}
                        ]
            },
            {
                "role": "assistant",
                "content": f'{{"reasoning": "{reasoning[ARTIFACT_TYPE]["positive"]["reasoning"]}", "success": true}}'
            },
            {
                "role": "user",
                "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{negative_original_masked}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{negative_original_target}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{negative_artifact_target}"}},
                            {"type": "text", "text": f"{reasoning[ARTIFACT_TYPE]['negative']['object_name']}"}
                        ]
            },
            {
                "role": "assistant",
                "content": f'{{"reasoning": "{reasoning[ARTIFACT_TYPE]["negative"]["reasoning"]}", "success": false}}'
            },
            {
                "role": "user",
                "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_original_masked}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_original_target}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_artifact_target}"}},
                            {"type": "text", "text": "a sheep and a sheep"}
                        ]
            }
        ],
        response_format={"type": "json_object"},
    )
    money_manager.refresh()
    money_manager(response)
    print(money_manager.total_cost)
    try:
        result = json.loads(response.choices[0].message.content)
        if result.get("success"):
            success_count += 1
    except (json.JSONDecodeError, KeyError, IndexError):
        pass

print(f"Success frequency: {success_count}/{total_runs}")

0.0138
0.01372
0.01383
0.01371
0.01379
0.01368
0.01368
0.01379
0.01374
0.01377
Success frequency: 7/10


# Without In-context learning template

In [ ]:
success_count = 0
total_runs = 10
for _ in range(total_runs):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_original_masked}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_original_target}"}},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{test_artifact_target}"}},
                            {"type": "text", "text": "a sheep and a sheep"}
                        ]
            }
        ],
        response_format={"type": "json_object"},
    )
    money_manager.refresh()
    money_manager(response)
    print(money_manager.total_cost)
    try:
        result = json.loads(response.choices[0].message.content)
        if result.get("success"):
            success_count += 1
    except (json.JSONDecodeError, KeyError, IndexError):
        pass

print(f"Success frequency: {success_count}/{total_runs}")

0.0048024999999999995
0.0049625
0.0047925
0.0046425
0.0047825
0.0047625
0.0048325
0.0048825
0.0049325
0.0048625
Success frequency: 9/10


In [ ]:
response.output_parsed.reasoning

'In the artifact region of the third image, the textures of the lamb and the sheep appear unnaturally merged. There is a noticeable lack of distinct separation between the two, with overlapping shadows and fur textures blending in a way that suggests fusion. The interface between the two animals has blurred boundaries, indicating an artifact.'